# Explore the SegTHOR data

An audit of **this local dataset**, from original 3D CT/label volumes to the 2D training
slices and saved model predictions. Counts are calculated from the files, not hardcoded.

**Start here:** open this notebook in VS Code, choose **Select Kernel → Python Environments →
`ai4mi/bin/python`**, then **Run All**. The first run reads all volumes and may take a few minutes.
Plots and interactive widgets appear inside this notebook; no XQuartz, Tkinter, GPU, or training job is needed.
If VS Code asks for notebook support, install its Jupyter extension on the SSH workspace.

The extra packages are listed in `requirements-notebooks.txt` at the project root. To install
them in a fresh environment: `python -m pip install -r requirements-notebooks.txt`.

What you will find:
1. Label meanings and file layout.
2. Split integrity, missing pairs, class imbalance, and per-patient statistics.
3. Original scan geometry, CT intensities, annotated volumes, and missing-label checks.
4. Representative examples and an interactive slice/prediction browser.
5. Learning curves and Dice calculations that expose the effect of empty classes.
6. CSV tables and PNG figures saved under `results/segthor/data_exploration/`.

The notebook reads the data and training results and writes only analysis exports.
It does not relabel masks or change the trained model. Organ names follow the repository's
README; numeric counts alone cannot establish whether that anatomical mapping is correct.

In [ ]:
from pathlib import Path
import os, sys, re, hashlib, json
from functools import lru_cache
from datetime import datetime, timezone

# Find the repository whether the kernel starts at its root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'slice_segthor.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Start the notebook from the ai4mi_project folder.')
DATA = ROOT / 'data/SEGTHOR'
RAW = ROOT / 'data/segthor_part1/train'
RUN = ROOT / 'results/segthor/ce'
EXPORT = ROOT / 'results/segthor/data_exploration'
EXPORT.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(EXPORT / '.mplconfig')
# Override any TkAgg setting left over from the SSH viewer troubleshooting.
os.environ.pop('MPLBACKEND', None)
import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display, Markdown
import ipywidgets as widgets
%matplotlib inline

RUN_RAW_AUDIT = True          # False skips the slower full-volume section.
VERIFY_ARCHIVE_SHA256 = False # Optional: reads the whole ZIP and verifies download integrity.
NAMES = ['Background', 'Esophagus', 'Heart', 'Trachea', 'Aorta']
COLORS = ['#252525', '#16a6e0', '#ed7d31', '#36a657', '#ce4a97']
CMAP = ListedColormap(COLORS)
NORM = BoundaryNorm(np.arange(-0.5, 5.5), CMAP.N)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

def save_figure(fig, name):
    fig.savefig(EXPORT / f'{name}.png', dpi=150, bbox_inches='tight')

def read_png(path):
    with Image.open(path) as im:
        return np.asarray(im).copy()

def decode_mask(path):
    mask = read_png(path)
    values = np.unique(mask)
    if mask.ndim != 2 or not np.isin(values, [0, 63, 126, 189, 252]).all():
        raise ValueError(f'Unexpected mask encoding: {path}: {values}')
    return (mask // 63).astype(np.uint8)

print('Python:', sys.executable)
print('Data:', DATA)
print('Analysis exports:', EXPORT)
print('Snapshot time (UTC):', datetime.now(timezone.utc).isoformat())

## 1. What the files and labels mean

`data/segthor_part1/train/Patient_XX/Patient_XX.nii.gz` is the original CT;
`GT.nii.gz` is its 3D annotation. Processed images and masks live in
`data/SEGTHOR/{train,val}/{img,gt}/Patient_XX_ZZZZ.png`, where `ZZZZ` is the
zero-based slice index along the original array's third axis.

`slice_segthor.py` normalizes **each entire CT volume** to 0–255 using its minimum and
maximum, resizes slices to 256 × 256 in the default Makefile, and resizes labels with
nearest-neighbor interpolation. It multiplies mask class IDs by 63 when writing PNGs.
`main.py` divides CT PNG values by 255 and mask values by 63 when loading them.
Processed intensities are **not Hounsfield units**. Pixel counts in resized PNGs are
also not physical organ volumes.

Background means all pixels without one of the foreground labels; it includes other
anatomy and any unannotated organs. Missing labels do not mean the anatomy is absent.

In [ ]:
class_key = pd.DataFrame({'class_id': range(5), 'name_from_readme': NAMES,
                          'raw_label_value': range(5), 'PNG_value': np.arange(5) * 63})
display(class_key)

## 2. Read every processed mask and check image/mask pairing

This audit records one row per mask, checks image dimensions and unexpected label values,
and hashes CT PNG contents to flag identical images appearing in both splits. Duplicate
images alone are not proof of patient leakage (for example, empty images can be identical).
Patients shared between training and validation are a stronger warning.

In [ ]:
rows, issues, pair_rows = [], [], []
for split in ['train', 'val']:
    images = {p.stem: p for p in (DATA / split / 'img').glob('*.png')}
    masks = {p.stem: p for p in (DATA / split / 'gt').glob('*.png')}
    pair_rows.append(dict(split=split, images=len(images), masks=len(masks),
                          images_without_mask=len(images.keys() - masks.keys()),
                          masks_without_image=len(masks.keys() - images.keys())))
    for stem in sorted(images.keys() ^ masks.keys()):
        issues.append(dict(split=split, file=stem, issue='Unpaired image/mask'))
    for stem, path in sorted(masks.items()):
        match = re.fullmatch(r'(Patient_\d+)_(\d+)', stem)
        if not match:
            issues.append(dict(split=split, file=stem, issue='Unexpected filename'))
            continue
        try:
            arr = read_png(path)
            mask = decode_mask(path)
            counts = np.bincount(mask.ravel(), minlength=5)
            rec = dict(split=split, patient=match[1], z=int(match[2]), stem=stem,
                       height=mask.shape[0], width=mask.shape[1], pixels=mask.size,
                       mask_path=str(path), image_hash=None)
            rec.update({f'pixels_{k}': int(counts[k]) for k in range(5)})
            rec['foreground_fraction'] = 1 - counts[0] / mask.size
            if stem in images:
                image = read_png(images[stem])
                if image.shape != mask.shape or image.ndim != 2:
                    issues.append(dict(split=split, file=stem, issue='Image/mask shape mismatch'))
                rec.update(image_path=str(images[stem]), image_mean=float(image.mean()),
                           image_std=float(image.std()), image_min=int(image.min()),
                           image_max=int(image.max()),
                           image_hash=hashlib.sha256(str(image.shape).encode() + image.tobytes()).hexdigest())
            rows.append(rec)
        except Exception as exc:
            issues.append(dict(split=split, file=stem, issue=str(exc)))
    print(f'{split}: {len(images)} images, {len(masks)} masks scanned', flush=True)
slices = pd.DataFrame(rows).sort_values(['split', 'stem']).reset_index(drop=True)
if slices.empty:
    raise RuntimeError(f'No valid masks found under {DATA}. Check the configuration cell.')
pairing = pd.DataFrame(pair_rows)
issue_table = pd.DataFrame(issues, columns=['split', 'file', 'issue'])
display(pairing)
display(issue_table if len(issue_table) else Markdown('No invalid masks or pairing problems found.'))

In [ ]:
split_summary = slices.groupby('split').agg(
    patients=('patient', 'nunique'), slices=('stem', 'size'), total_pixels=('pixels', 'sum'),
    mean_foreground_fraction=('foreground_fraction', 'mean'))
display(split_summary)
display(slices.groupby(['split', 'height', 'width']).size().rename('slice_count').to_frame())
train_patients = set(slices.loc[slices.split.eq('train'), 'patient'])
val_patients = set(slices.loc[slices.split.eq('val'), 'patient'])
shared_patients = sorted(train_patients & val_patients)
print('Train patients:', ', '.join(sorted(train_patients)))
print('Validation patients:', ', '.join(sorted(val_patients)))
print('Patients shared between splits:', shared_patients or 'None')
hash_splits = slices.dropna(subset=['image_hash']).groupby('image_hash')['split'].nunique()
cross_hashes = hash_splits[hash_splits > 1].index
cross_duplicates = slices[slices.image_hash.isin(cross_hashes)][['split', 'patient', 'stem', 'image_hash']]
print('Images in exact duplicate groups across splits:', len(cross_duplicates))
if len(cross_duplicates): display(cross_duplicates.head(20))

## 3. Class balance, patient coverage, and empty slices

**Pixel share** answers how much area is labelled. **Slices with class** and
**patients with class** answer how often it occurs. These are different: a small organ
may appear in many slices while occupying few pixels. Classes can coexist in a slice,
so class-presence percentages need not sum to 100%.

In [ ]:
class_rows = []
for split, group in slices.groupby('split'):
    for k, name in enumerate(NAMES):
        present = group[f'pixels_{k}'] > 0
        class_rows.append(dict(split=split, class_id=k, name=name,
            pixels=int(group[f'pixels_{k}'].sum()),
            pixel_share_pct=100 * group[f'pixels_{k}'].sum() / group.pixels.sum(),
            slices_with_class=int(present.sum()), slices_without_class=int((~present).sum()),
            slice_presence_pct=100 * present.mean(),
            patients_with_class=group.loc[present, 'patient'].nunique()))
class_stats = pd.DataFrame(class_rows)
display(class_stats)
patient_stats = slices.groupby(['split', 'patient']).agg(
    slices=('stem', 'size'), z_min=('z', 'min'), z_max=('z', 'max'),
    **{f'pixels_{k}': (f'pixels_{k}', 'sum') for k in range(5)})
patient_stats['missing_slice_indices'] = patient_stats.z_max - patient_stats.z_min + 1 - patient_stats.slices
display(patient_stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, split in enumerate(['train', 'val']):
    tab = class_stats[class_stats.split.eq(split)].set_index('class_id')
    axes[0].bar(np.arange(5) + (i - 0.5) * .36, tab.pixel_share_pct, .36, label=split)
    axes[1].bar(np.arange(4) + (i - 0.5) * .36, tab.loc[1:4, 'pixel_share_pct'], .36, label=split)
    axes[2].bar(np.arange(4) + (i - 0.5) * .36, tab.loc[1:4, 'slice_presence_pct'], .36, label=split)
for ax, names in zip(axes, [NAMES, NAMES[1:], NAMES[1:]]):
    ax.set_xticks(range(len(names)), names, rotation=35, ha='right')
    ax.legend()
axes[0].set(title='All pixels: background dominates', ylabel='% of pixels')
axes[1].set(title='Foreground classes only', ylabel='% of all pixels')
axes[2].set(title='Slices with a nonempty target', ylabel='% of slices', ylim=(0, 105))
fig.tight_layout(); save_figure(fig, 'class_balance'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
pat = patient_stats.reset_index()
patient_labels = pat.patient + ' (' + pat.split + ')'
presence_by_patient = np.array([
    [100 * (g[f'pixels_{k}'] > 0).mean() for k in range(1, 5)]
    for _, g in slices.groupby(['split', 'patient'])])
im = axes[0].imshow(presence_by_patient, vmin=0, vmax=100, cmap='Blues', aspect='auto')
axes[0].set_xticks(range(4), NAMES[1:])
axes[0].set_yticks(range(len(pat)), patient_labels)
axes[0].set_title('Per patient: % of slices containing each class')
fig.colorbar(im, ax=axes[0], label='% of patient slices')
for split, group in slices.groupby('split'):
    axes[1].hist(100 * group.foreground_fraction, bins=np.linspace(0, 20, 41),
                 alpha=.55, label=split)
axes[1].set(xlabel='Foreground area (% of pixels)', ylabel='Slices',
            title='Annotated foreground per slice (0–20% view)')
axes[1].legend()
fig.tight_layout(); save_figure(fig, 'patient_coverage'); plt.show()
display(slices.groupby('split').foreground_fraction.describe(percentiles=[.25, .5, .75, .95]))
print('Slices beyond the histogram range:', int((slices.foreground_fraction > .2).sum()))

## 4. Original 3D volumes: geometry, intensities, and label inventory

This reads every original CT and annotation one patient at a time. Label counts are exact.
Intensity percentiles and the histogram use a regular subsample of each CT (every fourth
voxel along each axis); minimum and maximum values use the full CT. Intensities are shown
in stored/scaled CT units, which are normally HU for CT, but this audit does not establish
their calibration independently.

Volumes in mL are calculated from the annotation affine determinant and spatial units.
If the NIfTI header says `unknown`, the table explicitly assumes millimeters, as the
repository's preprocessing does. Unexpected units give NaN rather than an invented volume.
Anatomical orientation comes from the affine; array views below are not a clinical viewer.

In [ ]:
raw_rows, raw_class_rows, raw_issues, intensity_samples = [], [], [], []
raw_files = sorted(RAW.glob('*/GT.nii.gz'))
if RUN_RAW_AUDIT:
    for gt_path in raw_files:
        patient = gt_path.parent.name
        try:
            scan_path = gt_path.parent / f'{patient}.nii.gz'
            scan = nib.load(scan_path)
            label = nib.load(gt_path)
            gt = np.asarray(label.dataobj)
            ct = np.asarray(scan.dataobj)
            if not np.isfinite(gt).all() or not np.equal(gt, np.floor(gt)).all() or gt.min() < 0 or gt.max() > 255:
                raise ValueError('Label volume contains nonfinite, noninteger or out-of-range values')
            counts = np.bincount(gt.astype(np.uint8).ravel(), minlength=256)
            values = np.flatnonzero(counts).tolist()
            spatial_unit = label.header.get_xyzt_units()[0]
            mm_per_unit = {'mm': 1, 'meter': 1000, 'micron': .001, 'unknown': 1}.get(spatial_unit, np.nan)
            voxel_ml = abs(np.linalg.det(label.affine[:3, :3])) * mm_per_unit**3 / 1000
            sample = ct[::4, ::4, ::4].ravel().astype(np.float32)
            finite = sample[np.isfinite(sample)]
            intensity_samples.append(finite[::max(1, len(finite) // 10000)])
            spacing = scan.header.get_zooms()[:3]
            selected = slices[slices.patient.eq(patient)]
            rec = dict(patient=patient, split=','.join(sorted(selected.split.unique())) or 'not processed',
                shape_x=ct.shape[0], shape_y=ct.shape[1], shape_z=ct.shape[2],
                spacing_x=float(spacing[0]), spacing_y=float(spacing[1]), spacing_z=float(spacing[2]),
                spatial_unit=spatial_unit, volume_assumes_mm=(spatial_unit == 'unknown'),
                orientation=''.join(nib.aff2axcodes(scan.affine)),
                CT_dtype=str(ct.dtype), GT_dtype=str(gt.dtype), label_values=str(values),
                shape_matches=(ct.shape == gt.shape), affine_matches=bool(np.allclose(scan.affine, label.affine)),
                finite_CT=bool(np.isfinite(ct).all()), intensity_min=float(np.nanmin(ct)),
                intensity_max=float(np.nanmax(ct)), sampled_p01=float(np.percentile(finite, 1)),
                sampled_median=float(np.median(finite)), sampled_p99=float(np.percentile(finite, 99)),
                processed_slices=len(selected), original_slices=ct.shape[2])
            raw_rows.append(rec)
            for k, name in enumerate(NAMES):
                raw_class_rows.append(dict(patient=patient, name=name, class_id=k,
                    voxels=int(counts[k]), volume_ml=float(counts[k] * voxel_ml),
                    slices_with_class=int(np.any(gt == k, axis=(0, 1)).sum()),
                    volume_assumes_mm=spatial_unit == 'unknown'))
            if not rec['shape_matches'] or not rec['affine_matches'] or not rec['finite_CT'] or any(v > 4 for v in values):
                raw_issues.append(dict(patient=patient, issue='Check shape, affine, finite CT, or unexpected label values'))
            print(f'{patient}: {ct.shape}, labels {values}', flush=True)
            del ct, gt, sample, finite
        except Exception as exc:
            raw_issues.append(dict(patient=patient, issue=str(exc)))
else:
    print('Raw-volume audit disabled in configuration.')
raw_stats = pd.DataFrame(raw_rows)
raw_classes = pd.DataFrame(raw_class_rows)
raw_issue_table = pd.DataFrame(raw_issues, columns=['patient', 'issue'])
if not raw_stats.empty:
    display(raw_stats)
    display(raw_stats[['spacing_x', 'spacing_y', 'spacing_z', 'shape_z', 'intensity_min', 'intensity_max']].describe())
if RUN_RAW_AUDIT and not raw_files:
    print('No original GT.nii.gz files found. Check RAW in the configuration.')
display(raw_issue_table if len(raw_issue_table) else Markdown('No raw-volume audit errors recorded.'))

In [ ]:
if not raw_classes.empty:
    raw_summary = raw_classes.groupby(['class_id', 'name']).agg(
        voxels=('voxels', 'sum'), patients_with_class=('voxels', lambda x: int((x > 0).sum())),
        median_volume_ml=('volume_ml', 'median'))
    display(raw_summary)
    raw_voxels = raw_classes.groupby('class_id').voxels.sum()
    processed_pixels = class_stats.groupby('class_id').pixels.sum()
    missing_check = class_key.copy()
    missing_check['raw_voxels'] = missing_check.class_id.map(raw_voxels)
    missing_check['processed_pixels'] = missing_check.class_id.map(processed_pixels)
    missing_check['interpretation'] = [
        'Absent in all original volumes checked' if r == 0 else
        'Present in raw, absent in processed: investigate slicing/split coverage' if p == 0 else
        'Present in raw and processed' for r, p in zip(missing_check.raw_voxels, missing_check.processed_pixels)]
    display(missing_check)
    original_ids = set(raw_stats.patient)
    print('Processed patients missing from raw audit:', sorted(set(slices.patient) - original_ids))
    print('Original patients absent from processed data:', sorted(original_ids - set(slices.patient)))
    print('Volume coverage: successfully audited', len(raw_stats), 'of', len(raw_files), 'original masks')
    display(raw_stats.loc[raw_stats.processed_slices.ne(raw_stats.original_slices),
                          ['patient', 'original_slices', 'processed_slices']])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for k in range(1, 5):
        volumes = raw_classes.loc[raw_classes.class_id.eq(k), 'volume_ml'].dropna()
        axes[0].scatter(np.full(len(volumes), k), volumes, color=COLORS[k], alpha=.65)
    axes[0].set_xticks(range(1, 5), NAMES[1:])
    axes[0].set(title='Annotated volume per patient (including zeros)', ylabel='mL (see spatial-unit assumptions)')
    axes[1].hist(np.concatenate(intensity_samples), bins=150, range=(-1200, 2000), color='#526a85')
    axes[1].set(title='Sampled raw CT intensity (display range −1200 to 2000)',
                xlabel='Stored/scaled CT intensity', ylabel='Sampled voxels')
    fig.tight_layout(); save_figure(fig, 'raw_volumes_and_intensity'); plt.show()

### Optional download-integrity check

Set `VERIFY_ARCHIVE_SHA256 = True` above to compare the ZIP with the repository checksum.
A matching hash verifies the archive bytes, not annotation completeness or anatomical
correctness. This check does not prove that the current extracted files came from that ZIP.
A recreated ZIP can have a different hash even when its contained files are unchanged.

In [ ]:
archive_check = {'status': 'not run'}
if VERIFY_ARCHIVE_SHA256:
    archive = ROOT / 'data/segthor_part1.zip'
    checksum = ROOT / 'data/segthor_part1.sha256'
    if archive.exists() and checksum.exists():
        digest = hashlib.sha256()
        with archive.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
                digest.update(chunk)
        expected = checksum.read_text().split()[0]
        archive_check = {'expected': expected, 'actual': digest.hexdigest(),
                         'matches': digest.hexdigest() == expected}
    else:
        archive_check = {'status': 'ZIP or checksum file missing'}
print(archive_check)

## 5. See the classes in the images

Each example below uses the validation slice with the largest annotated area for that
class, to make its label easy to see. These are **selected illustrations**, not random
examples or evidence of typical model quality. Colors are fixed across all figures.

In [ ]:
def show_ct(ax, image, contrast=True):
    low, high = np.percentile(image, [1, 99.5]) if contrast else (0, 255)
    ax.imshow(image, cmap='gray', vmin=low, vmax=max(high, low + 1))

def draw_overlay(ax, image, labels, selected=0, title='', alpha=.6, contrast=True):
    show_ct(ax, image, contrast)
    show = np.ma.masked_where((labels == 0) | ((labels != selected) if selected else False), labels)
    ax.imshow(show, cmap=CMAP, norm=NORM, interpolation='nearest', alpha=alpha)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

legend = [Patch(facecolor=COLORS[k], label=f'{k}: {NAMES[k]}') for k in range(1, 5)]
val_slices = slices[slices.split.eq('val')].sort_values('stem').reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for k in range(1, 5):
    examples = val_slices[val_slices[f'pixels_{k}'] > 0]
    if examples.empty:
        for ax in axes[:, k-1]:
            ax.text(.5, .5, f'{NAMES[k]}\nNo validation annotation', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
        continue
    row = examples.loc[examples[f'pixels_{k}'].idxmax()]
    image, mask = read_png(row.image_path), decode_mask(row.mask_path)
    show_ct(axes[0, k-1], image)
    axes[0, k-1].set_title(f'{NAMES[k]}: {row.stem}', fontsize=9); axes[0, k-1].axis('off')
    draw_overlay(axes[1, k-1], image, mask, selected=k, title=f'{int(row[f"pixels_{k}"]):,} annotated pixels')
fig.legend(handles=legend, loc='lower center', ncol=4)
fig.subplots_adjust(hspace=.25, wspace=.15, bottom=.10, top=.93)
save_figure(fig, 'class_examples'); plt.show()

### Interactive slice browser

Choose a patient, slice index, and class, then click **Show slice**. **All foreground** displays labels 1–4;
choosing one organ filters the overlay. The original CT remains visible behind masks.
Predictions are available only for validation patients with saved results. Red disagreement
pixels compare the two saved masks; they do not measure correctness of the source annotation.
Display contrast stretches each image's 1st–99.5th percentile by default; this changes only
the display, not stored pixel values. Uncheck **Contrast** to see the fixed 0–255 range.

The widgets need a live `ai4mi` kernel and VS Code's notebook widget renderer. Static examples
above remain visible without widgets. You can also call `view_slice(...)` directly in a new cell.

In [ ]:
def view_slice(split, patient, z, selected=0, alpha=.6, prediction='best_epoch', contrast=True):
    rows = slices[(slices.split == split) & (slices.patient == patient) & (slices.z == z)]
    if rows.empty:
        print('No slice at that index; choose a stored slice.'); return
    row = rows.iloc[0]
    image, gt = read_png(row.image_path), decode_mask(row.mask_path)
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    show_ct(axes[0], image, contrast)
    axes[0].set_title(f'{row.stem} — CT'); axes[0].axis('off')
    draw_overlay(axes[1], image, gt, selected, 'Ground truth', alpha, contrast)
    path = RUN / prediction / 'val' / f'{row.stem}.png'
    if split == 'val' and path.exists():
        pred = decode_mask(path)
        if pred.shape != gt.shape:
            raise ValueError('Prediction and ground-truth shapes differ.')
        draw_overlay(axes[2], image, pred, selected, prediction, alpha, contrast)
        error = (pred != gt) if selected == 0 else ((pred == selected) != (gt == selected))
        show_ct(axes[3], image, contrast)
        axes[3].imshow(np.ma.masked_where(~error, error), cmap=ListedColormap(['#ef3333']), alpha=.75)
        axes[3].set_title('Mask disagreement'); axes[3].axis('off')
    else:
        for ax in axes[2:]:
            ax.text(.5, .5, 'No saved prediction', ha='center', va='center', transform=ax.transAxes); ax.axis('off')
    fig.legend(handles=legend, loc='lower center', ncol=4)
    fig.tight_layout(rect=(0, .08, 1, 1)); plt.show()
    print('Ground-truth pixels:', {NAMES[k]: int((gt == k).sum()) for k in range(5)})

patient_options = [(f'{split}: {patient}', (split, patient))
                   for split, patient in sorted(set(zip(slices.split, slices.patient)))]
default_patient = ('val', sorted(val_slices.patient.unique())[0])
patient_widget = widgets.Dropdown(options=patient_options, value=default_patient, description='Patient')
z_widget = widgets.SelectionSlider(options=[0], description='Slice', continuous_update=False)
class_widget = widgets.Dropdown(options=[('All foreground', 0)] + [(NAMES[k], k) for k in range(1,5)], description='Class')
alpha_widget = widgets.FloatSlider(value=.6, min=0, max=1, step=.1, description='Opacity', continuous_update=False)
prediction_names = [p.name for p in sorted(RUN.glob('iter*')) if p.is_dir()]
if (RUN / 'best_epoch').is_dir(): prediction_names.insert(0, 'best_epoch')
pred_widget = widgets.Dropdown(options=prediction_names or ['best_epoch'], description='Prediction')
contrast_widget = widgets.Checkbox(value=True, description='Contrast')
def update_indices(change=None):
    split, patient = patient_widget.value
    indices = sorted(slices.loc[(slices.split == split) & (slices.patient == patient), 'z'].tolist())
    z_widget.options = indices
    z_widget.value = indices[len(indices)//2]
patient_widget.observe(update_indices, names='value')
update_indices()
def browse(patient_key, z, selected, alpha, prediction, contrast):
    view_slice(*patient_key, z, selected, alpha, prediction, contrast)
browser = widgets.interactive(browse, {'manual': True, 'manual_name': 'Show slice'},
    patient_key=patient_widget, z=z_widget, selected=class_widget, alpha=alpha_widget,
    prediction=pred_widget, contrast=contrast_widget)
display(browser)
# A static first view is also saved in the notebook, without requiring a widget click.
view_slice(*default_patient, z_widget.value)

In [ ]:
# Edit this patient to inspect where each organ appears along the volume.
PROFILE_PATIENT = sorted(val_slices.patient.unique())[0]
profile = val_slices[val_slices.patient.eq(PROFILE_PATIENT)].sort_values('z')
fig, ax = plt.subplots(figsize=(12, 4))
for k in range(1, 5):
    ax.plot(profile.z, profile[f'pixels_{k}'], color=COLORS[k], label=NAMES[k])
ax.set(title=f'{PROFILE_PATIENT}: annotated area along the scan',
       xlabel='Original array slice index', ylabel='Processed pixels per slice')
ax.legend(); fig.tight_layout(); save_figure(fig, 'patient_slice_profile'); plt.show()

## 6. Metrics: why an empty class can score 1

The training code (`utils.meta_dice`) uses `(2 × intersection + 1e-8) /
(target pixels + predicted pixels + 1e-8)`. If both masks are empty, its Dice is 1.
If an organ is absent on many slices, this convention can dominate the slice-average score.

The following audit recomputes scores directly from the saved best-epoch PNG predictions.
It reports:
* **Logged-style mean Dice:** includes empty target + empty prediction as 1.
* **Target-present mean Dice:** only slices containing that ground-truth class. This omits
  false positives on target-empty slices, so inspect their count too.
* **Pooled Dice / IoU:** sum intersections and areas over all slices before calculating a
  score, which gives large regions more weight and retains false positives on empty slices.
* **Per-patient pooled Dice:** treats each patient's full stack as a volume. Its equal-patient
  mean is different from the all-pixel pooled score and slice mean.

Classes with zero ground-truth support are reported as **NaN / unavailable** for supported
class summaries, even if the training convention gives 1. Excluding aorta alone does not
remove the empty-slice effect for other classes. These are validation metrics, not independent
test results; the best epoch was chosen using this same validation split.

In [ ]:
examples = pd.DataFrame([
    {'case': 'Both empty', 'target': 0, 'prediction': 0, 'intersection': 0},
    {'case': 'False positive on empty target', 'target': 0, 'prediction': 20, 'intersection': 0},
    {'case': 'Perfect nonempty overlap', 'target': 20, 'prediction': 20, 'intersection': 20},
    {'case': 'Partial overlap', 'target': 20, 'prediction': 20, 'intersection': 10}])
examples['training_Dice'] = (2 * examples.intersection + 1e-8) / (examples.target + examples.prediction + 1e-8)
display(examples)

In [ ]:
prediction_rows, prediction_issues = [], []
confusion = np.zeros((5, 5), dtype=np.int64)
pred_folder = RUN / 'best_epoch/val'
for row in val_slices.itertuples():
    path = pred_folder / f'{row.stem}.png'
    if not path.exists():
        prediction_issues.append({'stem': row.stem, 'issue': 'Missing prediction'}); continue
    try:
        gt, pred = decode_mask(row.mask_path), decode_mask(path)
        if gt.shape != pred.shape:
            raise ValueError('Prediction shape differs from ground truth')
        cm = np.bincount((5 * gt + pred).ravel(), minlength=25).reshape(5, 5)
        confusion += cm
        for k in range(5):
            target, predicted, intersection = int(cm[k].sum()), int(cm[:, k].sum()), int(cm[k, k])
            prediction_rows.append(dict(stem=row.stem, patient=row.patient, class_id=k,
                name=NAMES[k], target=target, predicted=predicted, intersection=intersection,
                dice=(2 * intersection + 1e-8) / (target + predicted + 1e-8)))
    except Exception as exc:
        prediction_issues.append({'stem': row.stem, 'issue': str(exc)})
predictions = pd.DataFrame(prediction_rows)
print('Audited prediction slices:', predictions.stem.nunique() if len(predictions) else 0, '/', len(val_slices))
if prediction_issues: display(pd.DataFrame(prediction_issues).head(20))
prediction_summary = pd.DataFrame()
patient_dice = pd.DataFrame()
if len(predictions):
    report = []
    for k, group in predictions.groupby('class_id'):
        t, p, inter = group[['target', 'predicted', 'intersection']].sum()
        supported = t > 0
        report.append(dict(class_id=k, name=NAMES[k], target_pixels=int(t), predicted_pixels=int(p),
            target_present_slices=int((group.target > 0).sum()),
            empty_empty_slices=int(((group.target == 0) & (group.predicted == 0)).sum()),
            false_positive_empty_slices=int(((group.target == 0) & (group.predicted > 0)).sum()),
            logged_style_mean=group.dice.mean(),
            target_present_mean=group.loc[group.target > 0, 'dice'].mean(),
            pooled_Dice=2 * inter / (t + p) if supported else np.nan,
            pooled_IoU=inter / (t + p - inter) if supported else np.nan,
            precision=inter / p if supported and p > 0 else np.nan,
            recall=inter / t if supported else np.nan))
    prediction_summary = pd.DataFrame(report)
    display(prediction_summary)
    patient_dice = predictions.groupby(['patient', 'class_id', 'name'])[['target', 'predicted', 'intersection']].sum().reset_index()
    patient_dice['pooled_Dice'] = np.where(patient_dice.target > 0,
        2 * patient_dice.intersection / (patient_dice.target + patient_dice.predicted).replace(0, np.nan), np.nan)
    display(patient_dice.pivot(index='patient', columns='name', values='pooled_Dice').reindex(columns=NAMES))
    display(patient_dice.groupby('name').pooled_Dice.agg(['mean', 'min', 'max']).reindex(NAMES))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    tab = prediction_summary.set_index('class_id').loc[1:4]
    for offset, column, label in [(-.25, 'logged_style_mean', 'All slices'),
                                (0, 'target_present_mean', 'Target-present slices'),
                                (.25, 'pooled_Dice', 'Pooled pixels')]:
        axes[0].bar(np.arange(4) + offset, tab[column], .25, label=label)
    axes[0].set_xticks(range(4), NAMES[1:]); axes[0].set(ylabel='Dice', ylim=(0, 1.05), title='Different averaging conventions')
    axes[0].legend(fontsize=8)
    norm_confusion = np.divide(confusion, confusion.sum(axis=1, keepdims=True),
                               out=np.full((5, 5), np.nan), where=confusion.sum(axis=1, keepdims=True) > 0)
    im = axes[1].imshow(np.ma.masked_invalid(norm_confusion), vmin=0, vmax=1, cmap='Blues')
    axes[1].set_xticks(range(5), NAMES, rotation=35, ha='right'); axes[1].set_yticks(range(5), NAMES)
    axes[1].set(xlabel='Predicted class', ylabel='Ground-truth class', title='Confusion: fractions within each target class')
    fig.colorbar(im, ax=axes[1]); fig.tight_layout(); save_figure(fig, 'prediction_metrics'); plt.show()
else:
    print('No usable saved predictions; data-only sections still work.')

### Learning curves and saved-log consistency

The notebook verifies saved best-epoch Dice against recomputed PNG Dice before matching
logged validation rows to target-presence masks. It assumes the repository's sorted,
unshuffled validation loader, then tests that assumption against saved predictions.
Training rows are shuffled each epoch, so target-present filtering is not applied to training logs.
All logged epochs are shown; all-zero rows are flagged as potentially unwritten. Epoch numbers
are zero-based, matching `iter000`, `iter001`, etc. Training loss curves average batch means.

In [ ]:
metric_paths = {name: RUN / f'{name}.npy' for name in ['dice_tra', 'dice_val', 'loss_tra', 'loss_val']}
metrics = {name: np.load(path, allow_pickle=False) for name, path in metric_paths.items() if path.exists()}
metric_summary = pd.DataFrame()
best_epoch = None
aligned = False
if (RUN / 'best_epoch.txt').exists():
    text = (RUN / 'best_epoch.txt').read_text()
    print(text)
    match = re.search(r'epoch (\d+):', text)
    if match: best_epoch = int(match[1])
if 'dice_val' in metrics:
    dv = metrics['dice_val']
    if dv.ndim != 3 or dv.shape[2] != 5:
        raise ValueError(f'Unexpected validation Dice array shape {dv.shape}')
    print('Validation log shape (epochs, slices, classes):', dv.shape)
    print('All-zero epoch rows (possibly not written):', np.flatnonzero(~dv.any(axis=(1, 2))).tolist())
    if len(predictions) and best_epoch is not None and best_epoch < len(dv):
        recomputed = predictions.pivot(index='stem', columns='class_id', values='dice').reindex(index=val_slices.stem, columns=range(5)).to_numpy()
        aligned = (len(val_slices) == dv.shape[1] and np.isfinite(recomputed).all()
                   and np.allclose(recomputed, dv[best_epoch], rtol=1e-5, atol=1e-6))
    print('Saved best-epoch scores match the current masks/predictions:', aligned)
    if not aligned:
        print('Target-present log curves skipped: use the recomputed prediction tables above; check that data/run match.')
    epochs = np.arange(len(dv))
    means = dv.mean(axis=1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    supported_classes = [k for k in range(1, 5) if val_slices[f'pixels_{k}'].sum() > 0]
    for k in range(1, 5):
        axes[0].plot(epochs, means[:, k], color=COLORS[k], label=f'{k}: {NAMES[k]}')
        if aligned and k in supported_classes:
            present = val_slices[f'pixels_{k}'].to_numpy() > 0
            axes[1].plot(epochs, dv[:, present, k].mean(axis=1), color=COLORS[k], label=NAMES[k])
    axes[0].plot(epochs, means[:, 1:].mean(axis=1), color='black', linestyle='--', label='Classes 1–4 mean')
    if supported_classes:
        axes[0].plot(epochs, means[:, supported_classes].mean(axis=1), color='#694fa8', linewidth=2.5, label='Supported organs mean')
    for ax, title in zip(axes[:2], ['Validation Dice: all slices', 'Validation Dice: target-present slices']):
        ax.set(xlabel='Epoch (zero-based)', ylabel='Dice', ylim=(0, 1.03), title=title)
        if ax.lines: ax.legend(fontsize=7)
        if best_epoch is not None: ax.axvline(best_epoch, color='gray', alpha=.4, linestyle=':')
    for name, label in [('loss_tra', 'Training'), ('loss_val', 'Validation')]:
        if name in metrics:
            loss = metrics[name]
            axes[2].plot(np.arange(len(loss)), loss.mean(axis=1), label=label)
    axes[2].set(title='Cross-entropy loss', xlabel='Epoch (zero-based)', ylabel='Mean batch loss')
    if axes[2].lines: axes[2].legend()
    fig.tight_layout(); save_figure(fig, 'learning_curves'); plt.show()
    metric_summary = pd.DataFrame({'epoch': epochs, **{f'dice_{NAMES[k].lower()}': means[:, k] for k in range(5)},
        'dice_all_5_classes': means.mean(axis=1), 'dice_organs_1_to_4': means[:, 1:].mean(axis=1)})
    if supported_classes:
        metric_summary['dice_supported_organs_all_slices'] = means[:, supported_classes].mean(axis=1)
    display(metric_summary.iloc[[best_epoch]] if best_epoch is not None and best_epoch < len(dv) else metric_summary.tail())
else:
    print('No validation metric log found at', metric_paths['dice_val'])

## 7. Export and interpret the findings

Tables below are exported as CSV; displayed static charts are also saved as PNG.
All outputs are a snapshot of the files read during this run. Rerun after replacing data
or training results. Do not interpret demographic distributions, diagnosis, scanner vendor,
annotation protocol, or intended missing labels from these arrays: those need dataset
documentation or confirmation from the course staff.

Read the audit in this order: (1) file/split integrity, (2) raw versus processed class presence,
(3) visual label alignment, (4) class balance, and only then (5) model metrics. An anatomically
wrong but numerically valid annotation will not necessarily fail an automated check.

In [ ]:
tables = {'slices': slices.drop(columns=['image_path', 'mask_path'], errors='ignore'),
          'split_summary': split_summary.reset_index(), 'pairing': pairing,
          'class_statistics': class_stats, 'patient_statistics': patient_stats.reset_index(),
          'data_issues': issue_table, 'cross_split_duplicate_images': cross_duplicates,
          'raw_scan_statistics': raw_stats, 'raw_class_volumes': raw_classes,
          'raw_issues': raw_issue_table, 'best_prediction_per_slice': predictions,
          'best_prediction_summary': prediction_summary, 'patient_dice': patient_dice,
          'epoch_metrics': metric_summary}
for name, table in tables.items():
    if len(table.columns): table.to_csv(EXPORT / f'{name}.csv', index=False)
pd.DataFrame(confusion, index=NAMES, columns=NAMES).to_csv(EXPORT / 'prediction_confusion_counts.csv')
(EXPORT / 'analysis_metadata.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(), 'dataset': str(DATA), 'raw': str(RAW),
    'run': str(RUN), 'class_mapping': dict(enumerate(NAMES)), 'raw_audit_enabled': RUN_RAW_AUDIT,
    'raw_volumes_successfully_checked': len(raw_stats), 'archive_check': archive_check,
    'best_epoch': best_epoch, 'saved_Dice_matches_current_masks': bool(aligned),
    'versions': {'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__, 'nibabel': nib.__version__}}, indent=2))
unsupported = [NAMES[k] for k in range(1, 5) if class_stats.loc[class_stats.class_id.eq(k), 'pixels'].sum() == 0]
print(f'{slices.patient.nunique()} patients; {len(slices):,} processed slices.')
print('Classes absent from all processed targets:', ', '.join(unsupported) or 'None')
print('Shared train/validation patients:', shared_patients or 'None')
print('Data issues:', len(issue_table), '| Raw audit issues:', len(raw_issue_table))
print('Saved all analysis tables and charts to:', EXPORT)
display(pd.DataFrame({'export': [p.name for p in sorted(EXPORT.glob('*')) if p.is_file()]}))